## Low Cost Pilot

In [9]:
import importlib,inspect,json,time
from pathlib import Path
import numpy as np,pandas as pd
from scipy.sparse import load_npz
import src.paths as paths,src.metrics as metrics,src.io_utils as io_utils
importlib.reload(paths);importlib.reload(metrics);importlib.reload(io_utils)

from src.paths import get_project_paths
from src.io_utils import load_embedding_conditions,load_id_file

P=get_project_paths();DATA=P["processed"];RESULTS=P["results"]
OUT=RESULTS/"week8"/"pilot";OUT.mkdir(parents=True,exist_ok=True)

PILOT_REPS=20;N_PERM=100;ALPHA=.05;BASE_SEED=42
SEVERITIES=[.10,.30,.60]

print("PILOT ONLY | reps =",PILOT_REPS)

PILOT ONLY | reps = 20


In [10]:
# Load the frozen 110-cell design
PREREG=RESULTS/"week8"/"preregistration"
anchors=pd.read_csv(PREREG/"n100_anchor_pruning_table.csv")
anchors.columns=[c.strip() for c in anchors.columns]

rows=[]
for n,col in [(50,"n=50 decision"),(200,"n=200 decision")]:
    x=anchors[anchors[col].astype(str).str.lower().str.contains("retain")].copy()
    x["N"]=n;rows.append(x)

pilot_design=pd.concat(rows,ignore_index=True)
pilot_design=pilot_design.rename(columns={"Test Configuration":"test","Scenario":"scenario","Severity":"severity"})
pilot_design["severity"]=pilot_design["severity"].astype(float).map(lambda x:x/100 if x>1 else x)

print("Retained pilot cells:",len(pilot_design))
display(pilot_design[["scenario","severity","test","N"]].head())
assert len(pilot_design)==110

Retained pilot cells: 110


,scenario,severity,test,N
0,Broken Timing,0.1,Energy,50
1,Broken Timing,0.1,MMD,50
2,Broken Timing,0.1,Sinkhorn,50
3,Broken Timing,0.3,C2ST-Linear,50
4,Broken Timing,0.3,C2ST-RBF,50


In [11]:
# Normalize names + find Sinkhorn implementation
def canon_scenario(x):
    x=str(x).lower()
    if "template" in x:return "Template Repetition"
    if "broken" in x:return "Broken Timing"
    if "entity" in x:return "Entity Inconsistency"
    if "missing" in x:return "Missing Cross-Channel Reference"
    if "polish" in x:return "Over-Polished Language"
    raise ValueError(x)

def canon_test(x):
    x=str(x).lower().replace("_","-")
    if "rbf" in x:return "C2ST-RBF"
    if "c2st" in x or "linear" in x:return "C2ST-Linear"
    if "mmd" in x:return "MMD"
    if "energy" in x:return "Energy"
    if "sink" in x:return "Sinkhorn"
    raise ValueError(x)

pilot_design["scenario"]=pilot_design["scenario"].map(canon_scenario)
pilot_design["test"]=pilot_design["test"].map(canon_test)

sink_candidates=[n for n in dir(metrics) if "sinkhorn" in n.lower() and callable(getattr(metrics,n))]
sink_candidates=sorted(sink_candidates,key=lambda n:("permutation" not in n.lower(),"paired" not in n.lower(),"test" not in n.lower()))
print("Sinkhorn candidates:",[(n,str(inspect.signature(getattr(metrics,n)))) for n in sink_candidates])

SINK=getattr(metrics,sink_candidates[0])
print("Selected:",SINK.__name__,inspect.signature(SINK))

Sinkhorn candidates: [('sinkhorn_paired_permutation_test', "(P, Q, blur, n_permutations=100, seed=42, alpha=0.05, p=2, backend='tensorized')"), ('sinkhorn_permutation_test', "(P, Q, blur, n_permutations=100, seed=42, alpha=0.05, p=2, backend='tensorized')"), ('_make_sinkhorn_loss', "(blur, p=2, backend='tensorized')"), ('_sinkhorn_value', '(P, Q, criterion)'), ('sinkhorn_divergence', "(P, Q, blur, p=2, backend='tensorized')")]
Selected: sinkhorn_paired_permutation_test (P, Q, blur, n_permutations=100, seed=42, alpha=0.05, p=2, backend='tensorized')


In [12]:
# Load frozen Template + semantic representation
entity=load_embedding_conditions(DATA,"entity",(10,30,60))
missing=load_embedding_conditions(DATA,"missing_reference",(10,30,60))
op=load_embedding_conditions(DATA,"over_polished",(10,30,60))

entity_ids=np.asarray(load_id_file(DATA,"entity_embedding_campaign_ids.csv","campaign_id")).astype(str)
missing_ids=np.asarray(load_id_file(DATA,"missing_reference_embedding_campaign_ids.csv","campaign_id")).astype(str)
op_ids=np.asarray(load_id_file(DATA,"over_polished_embedding_message_ids.csv","message_id")).astype(str)

tpl={"clean":load_npz(DATA/"template_tfidf_clean.npz").toarray()}
for s in SEVERITIES:tpl[s]=load_npz(DATA/f"template_tfidf_{int(s*100)}pct.npz").toarray()
tpl_ids=pd.read_csv(DATA/"template_repetition_formal_eval_ids.csv")["message_id"].astype(str).to_numpy()

for name,X,ids in [("Entity",entity,entity_ids),("Missing",missing,missing_ids),("OP",op,op_ids),("Template",tpl,tpl_ids)]:
    assert len(ids)==300
    assert all(np.asarray(X[k]).shape[0]==300 for k in ["clean",.10,.30,.60])
    print(name,{k:np.asarray(X[k]).shape for k in ["clean",.10,.30,.60]})

Entity {'clean': (300, 384), 0.1: (300, 384), 0.3: (300, 384), 0.6: (300, 384)}
Missing {'clean': (300, 384), 0.1: (300, 384), 0.3: (300, 384), 0.6: (300, 384)}
OP {'clean': (300, 384), 0.1: (300, 384), 0.3: (300, 384), 0.6: (300, 384)}
Template {'clean': (300, 5000), 0.1: (300, 5000), 0.3: (300, 5000), 0.6: (300, 5000)}


In [13]:
# Locate and load frozen Broken Timing matrices
BT_FEATURES=["n_messages","campaign_duration","mean_time_gap","median_time_gap","std_time_gap","n_channels","channel_switch_rate","channel_entropy","dominant_channel_share","mean_message_length","std_message_length","mean_token_count","std_token_count"]

bt_hits=[]
for root in [DATA,RESULTS/"week7",RESULTS]:
    for p in root.rglob("*broken_timing*formal*.csv"):
        try:
            d=pd.read_csv(p)
            if len(d)==300 and "campaign_id" in d and all(c in d for c in BT_FEATURES):
                A=d[BT_FEATURES].to_numpy(float)
                if np.isfinite(A).all() and np.max(np.abs(A))<30:bt_hits.append((p,d))
        except:pass

for p,_ in bt_hits:print(p)

In [17]:
BT_FEATURES=[
    "n_messages","campaign_duration","mean_time_gap","median_time_gap",
    "std_time_gap","n_channels","channel_switch_rate","channel_entropy",
    "dominant_channel_share","mean_message_length","std_message_length",
    "mean_token_count","std_token_count"
]

candidates=[]

for root in [DATA,RESULTS]:
    for p in root.rglob("*.csv"):
        try:
            d=pd.read_csv(p)
            if len(d)==300 and "campaign_id" in d.columns and all(c in d.columns for c in BT_FEATURES):
                A=d[BT_FEATURES].to_numpy(float)
                candidates.append({
                    "path":str(p),
                    "shape":A.shape,
                    "max_abs":float(np.max(np.abs(A))),
                    "first_id":str(d["campaign_id"].iloc[0]),
                    "last_id":str(d["campaign_id"].iloc[-1])
                })
        except:
            pass

cand=pd.DataFrame(candidates)
display(cand)
print("Candidates:",len(cand))

,path,shape,max_abs,first_id,last_id
0,/Users/jessica/fidelity-metric-lab/data/proces...,"(300, 13)",708.000000,C0003,C0999
1,/Users/jessica/fidelity-metric-lab/data/proces...,"(300, 13)",4.559492,C0003,C0999
2,/Users/jessica/fidelity-metric-lab/data/proces...,"(300, 13)",696.000000,C0003,C0999
3,/Users/jessica/fidelity-metric-lab/data/proces...,"(300, 13)",708.000000,C0003,C0999
4,/Users/jessica/fidelity-metric-lab/data/proces...,"(300, 13)",696.000000,C0003,C0999


Candidates: 5


In [ ]:
# Reconstruct and validate formal Broken Timing
import importlib,numpy as np,pandas as pd
from sklearn.preprocessing import StandardScaler
import src.defects as defects,src.representations as representations,src.metrics as metrics

importlib.reload(defects);importlib.reload(representations);importlib.reload(metrics)

from src.defects import make_broken_timing_plan,apply_broken_timing_plan
from src.representations import structural_campaign_summary
from src.metrics import median_heuristic_sigma

BT_FEATURES=[
    "n_messages","campaign_duration","mean_time_gap","median_time_gap",
    "std_time_gap","n_channels","channel_switch_rate","channel_entropy",
    "dominant_channel_share","mean_message_length","std_message_length",
    "mean_token_count","std_token_count"
]

clean_campaigns=pd.read_csv(DATA/"clean_structured_campaigns.csv",parse_dates=["timestamp"])
split_df=pd.read_csv(DATA/"structural_campaign_split.csv")

train_ids=set(split_df.loc[split_df["split"]=="train","campaign_id"])
eval_ids=set(split_df.loc[split_df["split"]=="evaluation","campaign_id"])

clean_train=clean_campaigns[clean_campaigns["campaign_id"].isin(train_ids)].copy()
clean_eval=clean_campaigns[clean_campaigns["campaign_id"].isin(eval_ids)].copy()

assert clean_train["campaign_id"].nunique()==700
assert clean_eval["campaign_id"].nunique()==300

plan=make_broken_timing_plan(
    clean_eval,
    severities=(.10,.30,.60),
    target_gap_minutes=-5.0,
    seed=42
)

def get_defective(severity):
    x=apply_broken_timing_plan(clean_eval,plan,severity)
    if isinstance(x,tuple):x=x[0]
    if isinstance(x,dict):
        x=x.get("defective",x.get("data",x))
    assert isinstance(x,pd.DataFrame)
    return x

conditions={"clean":clean_eval}
for s in [.10,.30,.60]:
    conditions[s]=get_defective(s)

train_raw=structural_campaign_summary(clean_train)
raw={k:structural_campaign_summary(v) for k,v in conditions.items()}

scaler=StandardScaler().fit(train_raw[BT_FEATURES])

bt_ids=raw["clean"]["campaign_id"].astype(str).to_numpy()
bt={}

for k,d in raw.items():
    aligned=d.set_index("campaign_id").loc[bt_ids]
    bt[k]=scaler.transform(aligned[BT_FEATURES])

assert len(bt_ids)==300
assert all(X.shape==(300,13) for X in bt.values())
assert all(np.isfinite(X).all() for X in bt.values())

changed={}
for s in [.10,.30,.60]:
    dist=np.linalg.norm(bt[s]-bt["clean"],axis=1)
    changed[s]=set(bt_ids[dist>1e-12])
    print(f"{s:.0%}: changed={len(changed[s])}, mean affected distance={dist[dist>1e-12].mean():.6f}")

assert [len(changed[s]) for s in [.10,.30,.60]]==[30,90,180]
assert changed[.10].issubset(changed[.30])
assert changed[.30].issubset(changed[.60])

train_scaled=scaler.transform(train_raw[BT_FEATURES])
bt_sigma=median_heuristic_sigma(train_scaled)

print("\nShapes:",{k:X.shape for k,X in bt.items()}) 
print("Frozen reconstructed MMD sigma:",bt_sigma)

expected_disp={.10:1.507846,.30:1.614294,.60:1.624370}

for s,target in expected_disp.items():
    dist=np.linalg.norm(bt[s]-bt["clean"],axis=1)
    observed=dist[dist>1e-12].mean()
    print(f"{s:.0%} | observed={observed:.6f} | Week7={target:.6f} | diff={abs(observed-target):.8f}")
    assert np.isclose(observed,target,atol=1e-5)

assert np.isclose(bt_sigma,4.829538,atol=1e-5)

print("\nBROKEN TIMING FORMAL RECONSTRUCTION PASSED")

10%: changed=30, mean affected distance=1.507845
30%: changed=90, mean affected distance=1.614294
60%: changed=180, mean affected distance=1.624370

Shapes: {'clean': (300, 13), 0.1: (300, 13), 0.3: (300, 13), 0.6: (300, 13)}
Frozen reconstructed MMD sigma: 4.829537929442184
10% | observed=1.507845 | Week7=1.507846 | diff=0.00000061
30% | observed=1.614294 | Week7=1.614294 | diff=0.00000033
60% | observed=1.624370 | Week7=1.624370 | diff=0.00000002

BROKEN TIMING FORMAL RECONSTRUCTION PASSED


The formal Broken Timing representation was successfully reconstructed. The expected 30, 90, and 180 affected campaigns were recovered at 10%, 30%, and 60% prevalence, respectively, and the clean and defective matrices all had the expected 300 × 13 shape. The reconstructed affected-unit displacement values matched the frozen Week 7 values to numerical precision, and the reconstructed MMD sigma matched the previously frozen value. This confirms that the pilot is using the same formal Broken Timing representation and defect construction as the original experiment

In [21]:
DATASETS={
    "Template Repetition":{"ids":tpl_ids,**tpl},
    "Broken Timing":{"ids":bt_ids,**bt},
    "Entity Inconsistency":{"ids":entity_ids,**entity},
    "Missing Cross-Channel Reference":{"ids":missing_ids,**missing},
    "Over-Polished Language":{"ids":op_ids,**op}
}

with open(RESULTS/"semantic_embedding_configuration.json") as f:sem_cfg=json.load(f)
with open(RESULTS/"over_polished_embedding_configuration.json") as f:op_cfg=json.load(f)

SCALES={
    "Template Repetition":{"sigma":1.4142135623730951,"blur":0.1414213562373095},
    "Broken Timing":{"sigma":4.829538,"blur":0.482954},
    "Entity Inconsistency":{"sigma":float(sem_cfg["mmd_sigma"]),"blur":0.043720},
    "Missing Cross-Channel Reference":{"sigma":float(sem_cfg["mmd_sigma"]),"blur":0.043720},
    "Over-Polished Language":{"sigma":float(op_cfg["mmd_sigma"]),"blur":0.132812}
}

display(pd.DataFrame(SCALES).T)

,sigma,blur
Template Repetition,1.414214,0.141421
Broken Timing,4.829538,0.482954
Entity Inconsistency,0.431204,0.043720
Missing Cross-Channel Reference,0.431204,0.043720
Over-Polished Language,1.328124,0.132812


The representation-specific MMD bandwidths and Sinkhorn blur values were successfully loaded for all five scenarios. Entity Inconsistency and Missing Cross-Channel Reference correctly share the same Campaign MiniLM scales because they use the same frozen representation. These parameters remain fixed throughout the pilot and are not adjusted using defective conditions.

In [22]:
# Deterministic shared sampling + metric runner
SCEN_IDX={s:i+1 for i,s in enumerate(DATASETS)}

def sample_pair(scenario,severity,n,rep):
    D=DATASETS[scenario];seed=BASE_SEED+SCEN_IDX[scenario]*100000+int(severity*1000)*100+n+rep
    idx=np.random.default_rng(seed).choice(300,n,replace=False)
    return np.asarray(D["clean"])[idx],np.asarray(D[severity])[idx],seed

def call_filtered(fn,P,Q,**kwargs):
    sig=inspect.signature(fn)
    return fn(P,Q,**{k:v for k,v in kwargs.items() if k in sig.parameters})

def run_metric(test,P,Q,scenario,seed):
    common={"test_size":.5,"n_permutations":N_PERM,"seed":seed,"alpha":ALPHA}
    if test=="C2ST-Linear":r=call_filtered(metrics.c2st_paired,P,Q,**common)
    elif test=="C2ST-RBF":r=call_filtered(metrics.c2st_rbf_paired,P,Q,C=1.0,gamma="scale",**common)
    elif test=="MMD":r=call_filtered(metrics.mmd_paired_permutation_test,P,Q,sigma=SCALES[scenario]["sigma"],**common)
    elif test=="Energy":r=call_filtered(metrics.energy_paired_permutation_test,P,Q,**common)
    elif test=="Sinkhorn":r=call_filtered(SINK,P,Q,blur=SCALES[scenario]["blur"],**common)
    else:raise ValueError(test)

    assert isinstance(r,dict) and "p_value" in r
    reject=r.get("reject_H0",r.get("reject"))
    stat=r.get("accuracy",r.get("statistic"))
    return {"statistic":stat,"p_value":r["p_value"],"reject":bool(reject),"null_mean":r.get("null_mean",np.nan),"null_std":r.get("null_std",np.nan)}

In [23]:
# One-repetition smoke test first
smoke=[]

for _,c in pilot_design.iterrows():
    P0,Q0,seed=sample_pair(c.scenario,c.severity,int(c.N),0)
    t=time.perf_counter()
    try:r=run_metric(c.test,P0,Q0,c.scenario,seed);err=""
    except Exception as e:r={"statistic":np.nan,"p_value":np.nan,"reject":False,"null_mean":np.nan,"null_std":np.nan};err=repr(e)
    smoke.append({**c[["scenario","severity","test","N"]].to_dict(),**r,"runtime":time.perf_counter()-t,"error":err})

smoke=pd.DataFrame(smoke)
print("Cells:",len(smoke),"Errors:",(smoke.error!="").sum(),"Nonfinite p:",smoke.p_value.isna().sum())
display(smoke[smoke.error!=""].head(20))

Cells: 110 Errors: 0 Nonfinite p: 0


,scenario,severity,test,N,statistic,p_value,reject,null_mean,null_std,runtime,error


The complete 110-cell smoke test passed without execution errors or non-finite p-values. This confirms that every preregistered retained Scenario × Severity × Test Configuration × N condition can be executed with the current frozen data, representations, parameters, and metric implementations.

In [24]:
# Run the 20-rep power pilot
records=[]
start=time.perf_counter()

for i,c in pilot_design.iterrows():
    for rep in range(PILOT_REPS):
        P0,Q0,seed=sample_pair(c.scenario,c.severity,int(c.N),rep)
        t=time.perf_counter();r=run_metric(c.test,P0,Q0,c.scenario,seed)
        records.append({"scenario":c.scenario,"severity":c.severity,"test":c.test,"N":int(c.N),"rep":rep,**r,"runtime":time.perf_counter()-t})
    if (i+1)%10==0:print(f"{i+1}/{len(pilot_design)} cells")

pilot_power=pd.DataFrame(records)
print("Rows:",len(pilot_power),"Expected:",110*PILOT_REPS)
print("Runtime:",round((time.perf_counter()-start)/60,2),"min")
assert len(pilot_power)==2200

10/110 cells
20/110 cells
30/110 cells
40/110 cells
50/110 cells
60/110 cells
70/110 cells
80/110 cells
90/110 cells
100/110 cells
110/110 cells
Rows: 2200 Expected: 2200
Runtime: 13.93 min


The low-cost defect-power pilot completed all 2,200 expected runs, corresponding to the 110 retained confirmatory cells with 20 Monte Carlo repetitions per cell. The full pilot required 13.93 minutes. These runs are used only to verify execution stability and computational feasibility; the observed rejection rates are not treated as confirmatory power estimates and are not used to modify the preregistered design.

In [25]:
# Save pilot power without using it for design changes
pilot_power.to_csv(OUT/"primary_power_pilot_20reps.csv",index=False)

pilot_power_summary=pilot_power.groupby(["scenario","severity","test","N"]).agg(
    reps=("reject","size"),rejections=("reject","sum"),pilot_rejection_rate=("reject","mean"),
    mean_p=("p_value","mean"),mean_runtime=("runtime","mean")
).reset_index()

pilot_power_summary.to_csv(OUT/"primary_power_pilot_20reps_summary.csv",index=False)
print("Saved:",len(pilot_power_summary),"cells")
assert len(pilot_power_summary)==110

Saved: 110 cells


In [26]:
# Build the 40 null-pilot cells
NULL_REPS={
    "TF-IDF":{"scenario":"Template Repetition"},
    "Structural":{"scenario":"Broken Timing"},
    "Campaign MiniLM":{"scenario":"Entity Inconsistency"},
    "Message MiniLM":{"scenario":"Over-Polished Language"}
}

tests=["C2ST-Linear","C2ST-RBF","MMD","Energy","Sinkhorn"]
null_design=pd.DataFrame([
    {"representation":rep,"scenario":v["scenario"],"test":test,"N":n}
    for rep,v in NULL_REPS.items() for test in tests for n in [50,200]
])

print("Null cells:",len(null_design))
assert len(null_design)==40

Null cells: 40


In [27]:
# Independent clean-vs-clean null pilot
def sample_null(scenario,n,rep):
    X=np.asarray(DATASETS[scenario]["clean"])
    base=BASE_SEED+SCEN_IDX[scenario]*200000+n*100+rep
    ia=np.random.default_rng(base).choice(300,n,replace=False)
    ib=np.random.default_rng(base+99991).choice(300,n,replace=False)
    return X[ia],X[ib],base

null_records=[]
start=time.perf_counter()

for i,c in null_design.iterrows():
    for rep in range(PILOT_REPS):
        P0,Q0,seed=sample_null(c.scenario,int(c.N),rep)
        t=time.perf_counter();r=run_metric(c.test,P0,Q0,c.scenario,seed)
        null_records.append({"representation":c.representation,"scenario":c.scenario,"test":c.test,"N":int(c.N),"rep":rep,**r,"runtime":time.perf_counter()-t})
    if (i+1)%10==0:print(f"{i+1}/{len(null_design)} null cells")

pilot_null=pd.DataFrame(null_records)
print("Rows:",len(pilot_null),"Expected:",40*PILOT_REPS)
print("Runtime:",round((time.perf_counter()-start)/60,2),"min")
assert len(pilot_null)==800

10/40 null cells
20/40 null cells
30/40 null cells
40/40 null cells
Rows: 800 Expected: 800
Runtime: 5.86 min


The clean-null pilot completed all 800 expected runs across the 40 planned n = 50 and n = 200 null-calibration cells. The null execution path was therefore operational across all four primary representation families and five test configurations. Pilot Type I rejection rates are not used to retune parameters or alter the confirmatory protocol.

In [28]:
# Save null pilot + operational validation
pilot_null.to_csv(OUT/"clean_null_pilot_20reps.csv",index=False)

pilot_null_summary=pilot_null.groupby(["representation","test","N"]).agg(
    reps=("reject","size"),rejections=("reject","sum"),pilot_type1=("reject","mean"),
    mean_p=("p_value","mean"),mean_runtime=("runtime","mean")
).reset_index()

pilot_null_summary.to_csv(OUT/"clean_null_pilot_20reps_summary.csv",index=False)

checks={
    "power_rows":len(pilot_power)==2200,
    "power_cells":len(pilot_power_summary)==110,
    "null_rows":len(pilot_null)==800,
    "null_cells":len(pilot_null_summary)==40,
    "power_finite_p":np.isfinite(pilot_power.p_value).all(),
    "null_finite_p":np.isfinite(pilot_null.p_value).all(),
    "power_reps_complete":pilot_power.groupby(["scenario","severity","test","N"]).size().eq(20).all(),
    "null_reps_complete":pilot_null.groupby(["representation","test","N"]).size().eq(20).all()
}

display(pd.Series(checks,name="passed"))
assert all(checks.values())
print("Operational pilot completed successfully.")

power_rows             True
power_cells            True
null_rows              True
null_cells             True
power_finite_p         True
null_finite_p          True
power_reps_complete    True
null_reps_complete     True
Name: passed, dtype: bool

Operational pilot completed successfully.


All predefined pilot integrity checks passed. The expected numbers of power and null runs were obtained, all p-values were finite, and every experimental cell contained the complete set of 20 pilot repetitions. No numerical or implementation failure required a change to the frozen metric suite, defect definitions, representation pipelines, pruning decisions, or representation-specific parameters. The low-cost primary-grid pilot is therefore operationally complete.

In [29]:
# Save null pilot + operational validation
pilot_null.to_csv(OUT/"clean_null_pilot_20reps.csv",index=False)

pilot_null_summary=pilot_null.groupby(["representation","test","N"]).agg(
    reps=("reject","size"),rejections=("reject","sum"),pilot_type1=("reject","mean"),
    mean_p=("p_value","mean"),mean_runtime=("runtime","mean")
).reset_index()

pilot_null_summary.to_csv(OUT/"clean_null_pilot_20reps_summary.csv",index=False)

checks={
    "power_rows":len(pilot_power)==2200,
    "power_cells":len(pilot_power_summary)==110,
    "null_rows":len(pilot_null)==800,
    "null_cells":len(pilot_null_summary)==40,
    "power_finite_p":np.isfinite(pilot_power.p_value).all(),
    "null_finite_p":np.isfinite(pilot_null.p_value).all(),
    "power_reps_complete":pilot_power.groupby(["scenario","severity","test","N"]).size().eq(20).all(),
    "null_reps_complete":pilot_null.groupby(["representation","test","N"]).size().eq(20).all()
}

display(pd.Series(checks,name="passed"))
assert all(checks.values())
print("Operational pilot completed successfully.")

power_rows             True
power_cells            True
null_rows              True
null_cells             True
power_finite_p         True
null_finite_p          True
power_reps_complete    True
null_reps_complete     True
Name: passed, dtype: bool

Operational pilot completed successfully.


The pilot run was successful and has already fulfilled its operational purpose. There is no technical reason in those outputs to modify the design, metrics, or representations.